# 03. EDA dan Data Cleaning IDM 2024
Notebook ini memuat eksplorasi awal, pembersihan data, penanganan outlier, dan ringkasan statistik sebelum proses clustering.

Hasil akhir notebook ini disimpan ke `data/idm_2024_clean.csv` untuk dipakai oleh notebook clustering.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', palette='muted')

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists() and (BASE_DIR.parent / 'data').exists():
    BASE_DIR = BASE_DIR.parent

RAW_DATA_PATH = BASE_DIR / 'data' / 'idm_2024_raw.csv'
CLEAN_DATA_PATH = BASE_DIR / 'data' / 'idm_2024_clean.csv'

print(f'Base dir   : {BASE_DIR}')
print(f'Raw data   : {RAW_DATA_PATH}')
print(f'Clean data : {CLEAN_DATA_PATH}')

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f'Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns')
display(df_raw.head(3))
df_raw.info()

In [ ]:
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_count', ascending=False)

if missing_df.empty:
    print('Tidak ada missing value pada dataset.')
else:
    display(missing_df)
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing_df) * 0.45)))
    missing_df['missing_pct'].plot(kind='barh', ax=ax, color='#4C78A8')
    ax.set_title('Missing Value per Kolom')
    ax.set_xlabel('Persentase (%)')
    plt.tight_layout()
    plt.show()

In [ ]:
df = df_raw.copy()

cols_to_drop = [c for c in ['Keterangan', 'Unnamed: 14'] if c in df.columns]
cols_to_drop.extend([c for c in df.columns if str(c).startswith('Unnamed:')])
cols_to_drop = sorted(set(cols_to_drop))
df = df.drop(columns=cols_to_drop, errors='ignore')
print('Dropped columns:', cols_to_drop)

df = df.drop_duplicates().reset_index(drop=True)

index_cols = ['IKS_2024', 'IKE_2024', 'IKL_2024', 'NILAI_IDM_2024']
mask_all_nan = df[index_cols].isna().all(axis=1)
df = df.loc[~mask_all_nan].reset_index(drop=True)

if 'NAMA_DESA' in df.columns:
    df['NAMA_DESA'] = df['NAMA_DESA'].fillna('Tidak_Diketahui')

for col in index_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

for col in ['KODE_PROV', 'KODE_KAB', 'KODE_KEC', 'KODE_DESA']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

status_map = {
    'SANGAT TERTINGGAL': 1,
    'TERTINGGAL': 2,
    'BERKEMBANG': 3,
    'MAJU': 4,
    'MANDIRI': 5,
}
if 'STATUS_IDM_2024' in df.columns:
    df['STATUS_IDM_2024'] = df['STATUS_IDM_2024'].astype(str).str.upper().map(status_map)

print(f'Cleaned data: {len(df):,} rows x {df.shape[1]} columns')
display(df.head(3))

In [ ]:
numeric_cols = ['IKS_2024', 'IKE_2024', 'IKL_2024', 'NILAI_IDM_2024']

def cap_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower=lower, upper=upper)

for col in numeric_cols:
    if col in df.columns:
        df[col] = cap_iqr(df[col])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color='#72B7B2')
    ax.set_title(col.replace('_2024', ''))
plt.tight_layout()
plt.show()

In [ ]:
if 'NAMA_PROVINSI' in df.columns:
    prov_summary = df.groupby('NAMA_PROVINSI')[numeric_cols].median().round(4)
    display(prov_summary.sort_values('NILAI_IDM_2024', ascending=False))

print('Ringkasan nasional:')
display(df[numeric_cols].describe().round(4))

df.to_csv(CLEAN_DATA_PATH, index=False, encoding='utf-8-sig')
print(f'Saved clean dataset to {CLEAN_DATA_PATH}')